# Caldwell Preserves Weekly Wildlife Survey — Colab

Load response data from the linked Google Sheet, inspect and clean it, run basic survey summaries, and export a CSV. Each response should represent one station visit.

## Access and privacy
Sign in to Google when prompted. The Google account used in Colab must have at least Viewer access to the response spreadsheet. This notebook reads the sheet through the authenticated Google Sheets API; it does not make the sheet public or store credentials in the notebook.

The survey may contain email addresses, names, exact GPS coordinates, and photograph links. The default export excludes columns whose headers look like personal identifiers or precise coordinates. Review the data and sharing permissions before distributing any export.

In [ ]:
%pip -q install gspread

from google.colab import auth
from google.auth import default
import gspread
import pandas as pd
import numpy as np
import re
from IPython.display import display
import matplotlib.pyplot as plt

auth.authenticate_user()
credentials, _ = default()
gc = gspread.authorize(credentials)
print('Google authentication complete.')

## Connect to the response spreadsheet
The spreadsheet ID is taken from the URL provided. The next cell lists available tabs and loads the first tab by default; change `WORKSHEET_NAME` to select another tab, such as a named form-response worksheet.

In [ ]:
SPREADSHEET_ID = '1ZqHQrlQWlGlUuywvK58DdK7PyVaBtmNFtFuehj13NeY'
WORKSHEET_NAME = None  # None selects the first tab; set to an exact tab name if needed.

spreadsheet = gc.open_by_key(SPREADSHEET_ID)
print('Spreadsheet:', spreadsheet.title)
print('Tabs:', [ws.title for ws in spreadsheet.worksheets()])
worksheet = spreadsheet.worksheet(WORKSHEET_NAME) if WORKSHEET_NAME else spreadsheet.get_worksheet(0)
values = worksheet.get_all_values()
if not values:
    raise ValueError('The selected worksheet is empty.')

def unique_headers(headers):
    seen = {}
    result = []
    for i, header in enumerate(headers):
        name = str(header).strip() or f'unnamed_{i + 1}'
        count = seen.get(name, 0)
        seen[name] = count + 1
        result.append(name if count == 0 else f'{name}_{count + 1}')
    return result

headers = unique_headers(values[0])
df = pd.DataFrame(values[1:], columns=headers)
df = df.replace(r'^\s*$', np.nan, regex=True).dropna(how='all').reset_index(drop=True)
print(f'Loaded {len(df):,} responses × {len(df.columns):,} columns from tab {worksheet.title!r}.')
display(df.head(3))

## Inspect and clean
Review the actual response headers before analysis. Google Forms can add wording or change header labels, so the summaries below locate columns by flexible name patterns rather than assuming exact header text.

In [ ]:
print('Columns:')
for col in df.columns:
    print('-', col)

quality = pd.DataFrame({
    'non_empty': df.notna().sum(),
    'missing': df.isna().sum(),
    'distinct_values': df.nunique(dropna=True),
    'example': [df[c].dropna().astype(str).head(1).tolist()[0] if df[c].notna().any() else '' for c in df.columns],
})
display(quality)

def find_columns(pattern):
    regex = re.compile(pattern, re.IGNORECASE)
    return [c for c in df.columns if regex.search(str(c))]

# Parse common date fields when present; keep the original text in df.
date_cols = find_columns(r'observation\s*date|^timestamp$|^date$')
clean_df = df.copy()
for col in date_cols:
    clean_df[col + '_parsed'] = pd.to_datetime(clean_df[col], errors='coerce')
print('Date-like columns:', date_cols or 'None detected')

## Quick survey summaries
These are exploratory summaries only. Check the detected columns and data entry conventions before drawing conclusions. For species checkbox questions, responses may be represented as multiple values in one cell or as separate columns; the code handles both basic formats.

In [ ]:
station_cols = find_columns(r'station\s*id|station')
if station_cols:
    station_col = station_cols[0]
    print('Responses by station:', station_col)
    display(clean_df[station_col].fillna('(missing)').value_counts().rename_axis('station').to_frame('responses'))

week_cols = find_columns(r'week\s*number|^week$')
if week_cols:
    week_col = week_cols[0]
    print('Responses by week:', week_col)
    display(clean_df[week_col].fillna('(missing)').value_counts().rename_axis('week').to_frame('responses'))

species_cols = find_columns(r'species observed|white.?tailed deer|squirrel|rabbit|raccoon|goose|mallard|robin|blackbird|woodpecker|hawk|crow|other bird|other mammal|other reptile|other amphibian')
if species_cols:
    print('Species/presence fields detected:', species_cols)
    # For one multi-select field, count selected labels; for separate fields, show non-empty response counts.
    if len(species_cols) == 1:
        selections = clean_df[species_cols[0]].dropna().astype(str).str.split(r'[,;\n]').explode().str.strip()
        selections = selections[selections.ne('')]
        display(selections.value_counts().rename_axis('selection').to_frame('responses'))
    else:
        display(clean_df[species_cols].notna().sum().sort_values(ascending=False).rename('non_empty responses').to_frame())
else:
    print('No species columns recognized automatically. Review the printed headers and select the appropriate columns manually.')

## Export a shareable CSV
By default, the export omits email/name fields, precise latitude/longitude, and columns that appear to contain photo uploads. Change the flag only if your data-handling plan allows it. The raw `df` remains unchanged.

In [ ]:
INCLUDE_SENSITIVE_COLUMNS = False

def is_sensitive_header(header):
    text = str(header).lower()
    patterns = [r'\bemail\b', r'observer\s*name', r'\bname\b', r'latitude', r'longitude', r'\bgps\b', r'photograph', r'photo', r'file upload']
    return any(re.search(p, text) for p in patterns)

export_df = clean_df.copy()
if not INCLUDE_SENSITIVE_COLUMNS:
    export_df = export_df[[c for c in export_df.columns if not is_sensitive_header(c)]]

output_path = 'caldwell_preserves_wildlife_observations.csv'
export_df.to_csv(output_path, index=False)
print(f'Saved {len(export_df):,} records and {len(export_df.columns):,} columns to {output_path}')

# Uncomment in Colab to download the CSV to your computer.
# from google.colab import files
# files.download(output_path)

## Next steps
1. Run cells from top to bottom and sign in with an account that can access the response sheet.
2. Confirm that the loaded tab is the form's response tab and inspect the headers.
3. Verify units, missing-value conventions, checkbox formats, and station labels before interpreting results.
4. Keep raw exports and precise location/photo data restricted to authorized collaborators.